In [ ]:
import os

import torch
import torch.nn as nn
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from data_setup import TimTremorSensor, TimTremorGraph, TimTremor, TRANSFORM
from model_builder import *
import csv
from datetime import datetime
import json
import pandas as pd
import gc

from torchinfo import summary
from torchmetrics.classification import Accuracy, Precision, Recall, ConfusionMatrix, F1Score
from tqdm.auto import tqdm
from timeit import default_timer as timer 

try:
    from sklearn.utils import resample
    from sklearn.metrics import f1_score
except:
    # %pip install scikit-learn
    from sklearn.utils import resample
    from sklearn.metrics import f1_score

In [ ]:
HYPER_PARAMS = {
    'K': 3,
    'LR': 1e-3,
    'BATCH_SIZE': 16,
    'EPOCHS': 1
}

In [ ]:
class CONFIG:
    num_classes = 2
    dropout = 0.1
    final_embd_dim = 384
    nhead = 8
    num_encoder_layers = 6
    hidden_size = 768
    graph_pretrained = os.path.join("graph_pretrained.pth")
    sensor_pretrained = os.path.join("sensor_pretrained.pth")
    graph_sensor_pretrained = os.path.join("graph_sensor_pretrained.pth")
    graph_freeze = False
    sensor_freeze = False
    graph_sensor_freeze = False
    device = None

In [ ]:
class PerformanceMetric:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.accuracy = Accuracy(task="multiclass", num_classes=num_classes)
        self.precision = Precision(task="multiclass", num_classes=num_classes, average='macro')
        self.recall = Recall(task="multiclass", num_classes=num_classes, average='macro')
        self.f1_weighted = F1Score(task="multiclass", num_classes=num_classes, average='weighted')
        self.f1_macro = F1Score(task="multiclass", num_classes=num_classes, average='macro')
        self.confusion_matrix = ConfusionMatrix(task="multiclass", num_classes=num_classes)

        self.predictions = []
        self.targets = []
        

    def update(self, predictions, targets):
        self.accuracy.update(predictions, targets)
        self.precision.update(predictions, targets)
        self.recall.update(predictions, targets)
        self.f1_weighted.update(predictions, targets)
        self.f1_macro.update(predictions, targets)
        self.confusion_matrix.update(predictions, targets)

        self.predictions = self.predictions + predictions.tolist()
        self.targets = self.targets + targets.tolist()

    def compute(self):
        accuracy = self.accuracy.compute().item()
        precision = self.precision.compute().item()
        recall = self.recall.compute().item()
        f1_weighted = self.f1_weighted.compute().item()
        f1_macro = self.f1_macro.compute().item()
        confusion_matrix = self.confusion_matrix.compute().cpu()

        return accuracy, precision, recall, f1_weighted, f1_macro, confusion_matrix
    
    def calculate_bootstrap_confidence_interval(self):
        if len(self.predictions) == 0 or len(self.targets) == 0:
            return [], 0, 0, 0

        y_true = np.array(self.targets)
        y_pred = np.array(self.predictions)        


        n_iter = 1000
        n_samples = len(y_true)
        stats = []

        for _ in range(n_iter):
            indices = resample(np.arange(n_samples), replace=True)

            sample_preds = y_pred[indices]
            sample_targets = y_true[indices]

            score = f1_score(sample_targets, sample_preds, average='weighted')
            stats.append(score)

        alpha = 0.95
        p = ((1.0 - alpha) / 2.0) * 100
        lower = np.percentile(stats, p)
        upper = np.percentile(stats, 100-p)
        mean = np.mean(stats)
        
        return stats, lower, mean, upper

    def reset(self):
        self.accuracy.reset()
        self.precision.reset()
        self.recall.reset()
        self.f1_macro.reset()
        self.f1_weighted.reset()
        self.confusion_matrix.reset()

        self.predictions = []
        self.targets = []


In [ ]:
def graph_sensor_collate_fn(batch):
    landmarks = []
    graph_types = []
    accs = []
    labels = []

    for item in batch:
        landmark, graph_type, acc, label = item
        landmarks.append(landmark)
        graph_types.append(graph_type)
        accs.append(acc)
        labels.append(label)

    landmarks = torch.stack(landmarks)
    graph_types = torch.tensor(graph_types)
    accs = torch.stack(accs)
    labels = torch.tensor(labels)

    return landmarks, graph_types, accs, labels

def graph_collate_fn(batch):
    landmarks = []
    graph_types = []
    labels = []

    for item in batch:
        landmark, graph_type, label = item
        landmarks.append(landmark)
        graph_types.append(graph_type)
        labels.append(label)

    landmarks = torch.stack(landmarks)
    graph_types = torch.tensor(graph_types)
    labels = torch.tensor(labels)

    return landmarks, graph_types, labels

In [ ]:
class Tester:
    def __init__(self,
                 data_root_dir:str,
                 test_instance:str,
                 transform: transforms.Compose=TRANSFORM,
                 model_config: CONFIG = CONFIG,
                 hyper_params: dict = HYPER_PARAMS
                 ):
        
        # data directories
        self.data_root_dir = data_root_dir
        self.test_instance = test_instance
        self.transform = transform
        self.model_config = model_config
        self.BATCH_SIZE = hyper_params['BATCH_SIZE']

        # result directories
        os.makedirs('TEST', exist_ok=True)
        self.result_root_dir = os.path.join('TEST', datetime.now().strftime("%Y-%m-%d_%H-%M-%S"))
        os.makedirs(self.result_root_dir, exist_ok=True)
    
    def setup_device(self):
        # check if CUDA is available
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        print("[INFO] Using device: ", self.device)
        self.model_config.device = self.device

    def setup_seed(self, seed=42):
        # set up the seed for reproducibility
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.cuda.manual_seed(seed)

    def create_dataloaders(self, instance:str, flag:str, shuffle=True):
        if flag == 'graph':
            data = TimTremorGraph(self.data_root_dir, instance, self.transform)   
            data_loader = DataLoader(data,
                                    batch_size=self.BATCH_SIZE,
                                    shuffle=shuffle,
                                    num_workers=0,
                                    pin_memory=False,
                                    collate_fn=graph_collate_fn)
        elif flag == 'sensor':
            data = TimTremorSensor(self.data_root_dir, instance, self.transform)   
            data_loader = DataLoader(data,
                                    batch_size=self.BATCH_SIZE,
                                    shuffle=shuffle,
                                    num_workers=0,
                                    pin_memory=False)
        elif flag == 'graph_sensor':
            data = TimTremor(self.data_root_dir, instance, self.transform)
            data_loader = DataLoader(data,
                                    batch_size=self.BATCH_SIZE,
                                    shuffle=shuffle,
                                    num_workers=0,
                                    pin_memory=False,
                                    collate_fn=graph_sensor_collate_fn)

        class_names = data.classes

        return data_loader, class_names
    
    def setup_dataloaders(self):
        self.test_dataloader = {}

        for key, value in self.test_instance.items():
            test = []
            for val in value:
                data_loader, class_names = self.create_dataloaders(val, key, shuffle=False)
                test.append(data_loader)
            self.test_dataloader[key] = test
            self.class_names = class_names

        self.num_classes = len(self.class_names)
        self.num_classes = 2

    def setup_model(self):   
        self.multimodal_model = ModelV2(self.model_config)
        if self.model_config.graph_sensor_pretrained is not None:
            self.multimodal_model.load_state_dict(torch.load(self.model_config.graph_sensor_pretrained, map_location=self.model_config.device))
        self.multimodal_model.to(self.device)

    def setup_loss_function(self):
        self.graph_loss_fn = nn.CrossEntropyLoss()
        self.sensor_loss_fn = nn.CrossEntropyLoss()
        self.multimodal_loss_fn = nn.CrossEntropyLoss()
    
    def test_step(self):
        self.multimodal_model.eval()

        graph_test_loss = 0
        sensor_test_loss = 0
        multimodal_test_loss = 0

        graph_metrics = PerformanceMetric(self.num_classes)
        sensor_metrics = PerformanceMetric(self.num_classes)
        multimodal_metrics = PerformanceMetric(self.num_classes)

        total_data_instance = 0

        with torch.inference_mode():
            for key, value in self.test_dataloader.items():
                for test_dataloader in value:
                    total_data_instance += len(test_dataloader)

                    if key == 'graph':
                        for batch, (X1, graph_type, y) in enumerate(test_dataloader):
                            X1, graph_type, y = X1.to(self.device), graph_type.to(self.device), y.to(self.device)
                            X2 = None
                            y_pred, graph_pred, sensor_pred, _, _ = self.multimodal_model(X1, X2, graph_type)

                            graph_test_loss += self.graph_loss_fn(graph_pred, y).item()
                            multimodal_test_loss += self.multimodal_loss_fn(y_pred, y).item()

                            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
                            multimodal_metrics.update(y_pred_class.cpu(), y.cpu())

                            y_pred_class = torch.argmax(torch.softmax(graph_pred, dim=1), dim=1)
                            graph_metrics.update(y_pred_class.cpu(), y.cpu())

                    elif key == 'sensor':
                        for batch, (X2, y) in enumerate(test_dataloader):
                            X2, y = X2.to(self.device), y.to(self.device)
                            X1 = None
                            graph_type = None
                            y_pred, graph_pred, sensor_pred, _, _ = self.multimodal_model(X1, X2, graph_type)

                            sensor_test_loss += self.sensor_loss_fn(sensor_pred, y).item()
                            multimodal_test_loss += self.multimodal_loss_fn(y_pred, y).item()

                            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
                            multimodal_metrics.update(y_pred_class.cpu(), y.cpu())

                            y_pred_class = torch.argmax(torch.softmax(sensor_pred, dim=1), dim=1)
                            sensor_metrics.update(y_pred_class.cpu(), y.cpu())

                    elif key == 'graph_sensor':
                        for batch, (X1, graph_type, X2, y) in enumerate(test_dataloader):
                            X1, X2, graph_type, y = X1.to(self.device), X2.to(self.device), graph_type.to(self.device), y.to(self.device)
                            y_pred, graph_pred, sensor_pred, _, _ = self.multimodal_model(X1, X2, graph_type)

                            graph_test_loss += self.graph_loss_fn(graph_pred, y).item()
                            sensor_test_loss += self.sensor_loss_fn(sensor_pred, y).item()
                            multimodal_test_loss += self.multimodal_loss_fn(y_pred, y).item()
                            
                            graph_pred_class = torch.argmax(torch.softmax(graph_pred, dim=1), dim=1)
                            graph_metrics.update(graph_pred_class.cpu(), y.cpu())

                            sensor_pred_class = torch.argmax(torch.softmax(sensor_pred, dim=1), dim=1)
                            sensor_metrics.update(sensor_pred_class.cpu(), y.cpu())

                            y_pred_class = torch.argmax(torch.softmax(y_pred, dim=1), dim=1)
                            multimodal_metrics.update(y_pred_class.cpu(), y.cpu())                    

        if total_data_instance == 0:
            total_data_instance = 1

        graph_test_loss /= total_data_instance
        sensor_test_loss /= total_data_instance
        multimodal_test_loss /= total_data_instance

        graph_accuracy, graph_precision, graph_recall, graph_f1_weighted, graph_f1_macro, graph_conf_matrix = graph_metrics.compute()
        sensor_accuracy, sensor_precision, sensor_recall, sensor_f1_weighted, sensor_f1_macro, sensor_conf_matrix = sensor_metrics.compute()
        multimodal_accuracy, multimodal_precision, multimodal_recall, multimodal_f1_weighted, multimodal_f1_macro, multimodal_conf_matrix = multimodal_metrics.compute()
        
        graph_stats, graph_lower, graph_mean, graph_upper = graph_metrics.calculate_bootstrap_confidence_interval()
        sensor_stats, sensor_lower, sensor_mean, sensor_upper = sensor_metrics.calculate_bootstrap_confidence_interval()
        multimodal_stats, multimodal_lower, multimodal_mean, multimodal_upper = multimodal_metrics.calculate_bootstrap_confidence_interval()

        graph_metrics.reset()
        sensor_metrics.reset()
        multimodal_metrics.reset()

        return graph_test_loss, sensor_test_loss, multimodal_test_loss, graph_accuracy, graph_precision, graph_recall, graph_f1_weighted, graph_f1_macro, graph_conf_matrix, sensor_accuracy, sensor_precision, sensor_recall, sensor_f1_weighted, sensor_f1_macro, sensor_conf_matrix, multimodal_accuracy, multimodal_precision, multimodal_recall, multimodal_f1_weighted, multimodal_f1_macro, multimodal_conf_matrix, graph_stats, graph_lower, graph_mean, graph_upper, sensor_stats, sensor_lower, sensor_mean, sensor_upper, multimodal_stats, multimodal_lower, multimodal_mean, multimodal_upper

    def run(self):
        # set up the device
        self.setup_device()

        # set up the seed
        self.setup_seed()

        # set up the data loaders
        self.setup_dataloaders()

        # set up the model
        self.setup_model()

        # set up the loss function
        self.setup_loss_function()

        # test the model
        graph_test_loss, sensor_test_loss, multimodal_test_loss, graph_accuracy, graph_precision, graph_recall, graph_f1_weighted, graph_f1_macro, graph_conf_matrix, sensor_accuracy, sensor_precision, sensor_recall, sensor_f1_weighted, sensor_f1_macro, sensor_conf_matrix, multimodal_accuracy, multimodal_precision, multimodal_recall, multimodal_f1_weighted, multimodal_f1_macro, multimodal_conf_matrix, graph_stats, graph_lower, graph_mean, graph_upper, sensor_stats, sensor_lower, sensor_mean, sensor_upper, multimodal_stats, multimodal_lower, multimodal_mean, multimodal_upper = self.test_step()
        path = os.path.join(self.result_root_dir, 'test_metrics.txt')

        with open(path, 'w') as f:
            f.write(f"Graph Test Loss: {graph_test_loss}\n")
            f.write(f"Graph Test Accuracy: {graph_accuracy}\n")
            f.write(f"Graph Test Precision: {graph_precision}\n")
            f.write(f"Graph Test Recall: {graph_recall}\n")
            f.write(f"Graph Test F1 (Weighted): {graph_f1_weighted}\n")
            f.write(f"Graph Test F1 (Macro): {graph_f1_macro}\n")
            f.write(f"Graph Test Confusion Matrix:\n{graph_conf_matrix.numpy()}\n")
            f.write(f"Sensor Test Loss: {sensor_test_loss}\n")
            f.write(f"Sensor Test Accuracy: {sensor_accuracy}\n")
            f.write(f"Sensor Test Precision: {sensor_precision}\n")
            f.write(f"Sensor Test Recall: {sensor_recall}\n")
            f.write(f"Sensor Test F1 (Weighted): {sensor_f1_weighted}\n")
            f.write(f"Sensor Test F1 (Macro): {sensor_f1_macro}\n")
            f.write(f"Sensor Test Confusion Matrix:\n{sensor_conf_matrix.numpy()}\n")
            f.write(f"Multimodal Test Loss: {multimodal_test_loss}\n")
            f.write(f"Multimodal Test Accuracy: {multimodal_accuracy}\n")
            f.write(f"Multimodal Test Precision: {multimodal_precision}\n")
            f.write(f"Multimodal Test Recall: {multimodal_recall}\n")
            f.write(f"Multimodal Test F1 (Weighted): {multimodal_f1_weighted}\n")
            f.write(f"Multimodal Test F1 (Macro): {multimodal_f1_macro}\n")
            f.write(f"Multimodal Test Confusion Matrix:\n{multimodal_conf_matrix.numpy()}\n")
            f.write(f"Graph Bootstrap Lower Bound: {graph_lower}\n")
            f.write(f"Graph Bootstrap Mean: {graph_mean}\n")
            f.write(f"Graph Bootstrap Upper Bound: {graph_upper}\n")
            f.write(f"Sensor Bootstrap Lower Bound: {sensor_lower}\n")
            f.write(f"Sensor Bootstrap Mean: {sensor_mean}\n")
            f.write(f"Sensor Bootstrap Upper Bound: {sensor_upper}\n")
            f.write(f"Multimodal Bootstrap Lower Bound: {multimodal_lower}\n")
            f.write(f"Multimodal Bootstrap Mean: {multimodal_mean}\n")
            f.write(f"Multimodal Bootstrap Upper Bound: {multimodal_upper}\n")

        graph_stats_path = os.path.join(self.result_root_dir, 'graph_bootstrap_stats.txt')
        with open(graph_stats_path, 'w') as f:
            for stat in graph_stats:
                f.write(f"{stat}\n")
        sensor_stats_path = os.path.join(self.result_root_dir, 'sensor_bootstrap_stats.txt')
        with open(sensor_stats_path, 'w') as f:
            for stat in sensor_stats:
                f.write(f"{stat}\n")
        multimodal_stats_path = os.path.join(self.result_root_dir, 'multimodal_bootstrap_stats.txt')
        with open(multimodal_stats_path, 'w') as f:
            for stat in multimodal_stats:
                f.write(f"{stat}\n")    
        

In [ ]:
data_root_dir = "TimTremorMultimodal"

test_instance_path = {
    'sensor': ['sensor_test.csv', 'graph_sensor_test.csv', 'graph_sensor_test.csv']
}

config = CONFIG()

tester = Tester(data_root_dir=data_root_dir,
                  test_instance=test_instance_path,
                  model_config=config)
tester.run()